### SetUp

In [ ]:
from project_modules.project_imports import *
from pathlib import Path

from project_modules.classes import Patient
from project_modules.classes import Clinic

from project_modules.rules import call_a_rule

from project_modules.simulation_tools import create_appointments, plot_line_graph
from project_modules.simulation_tools import establish_attendance, random_patient_sample
from project_modules.simulation_tools import get_margin_errors, check_convergence_mean, confidence_interval, calculate_summary

from openpyxl.utils import get_column_letter

In [ ]:
project_path = Path.cwd()
print(project_path)

In [ ]:
def safe_divide(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

def confidence_bounds(mean_value, margin):
    return mean_value - margin, mean_value + margin

def run_metric_tests(series_a, series_b, metric_name):
    """
    Runs 4 t-tests:
        greater / two-sided
        welch correction True / False
    Returns dataframe.
    """
    tests = []

    configs = [
        ("greater", True),
        ("greater", False),
        ("two-sided", True),
        ("two-sided", False),
    ]

    for alternative, correction in configs:

        df = ttest(
            series_a,
            series_b,
            correction=correction,
            alternative=alternative,
            confidence=0.95
        )

        tests.append({
            "metric": metric_name,
            "alternative": alternative,
            "welch_correction": correction,
            "t_stat": df["T"].values[0],
            "p_value": df["p_val"].values[0]
        })

    return pd.DataFrame(tests)

def autosize_excel_columns(writer, sheet_name):
    ws = writer.sheets[sheet_name]

    for col_cells in ws.columns:
        max_len = max(
            len(str(cell.value)) if cell.value is not None else 0
            for cell in col_cells
        )
        col_letter = get_column_letter(col_cells[0].column)
        ws.column_dimensions[col_letter].width = max(max_len + 2, 12)

### Parameter definition

In [ ]:
# ============================================================
# #                   CLINIC CONFIGURATION                   #
# ============================================================

# Configuration Constants
SEED = 42
SLOT_DURATION_MINS = 20
NUM_SERVERS = 1
OPERATING_HOURS = 10
SIMULATION_DAYS = 7
EXTRA_DEMAND_RATE = 0.35 # The percentage of extra patients to simulate beyond capacity

# Derived calculations
SLOTS_PER_DAY = (60 // SLOT_DURATION_MINS) * OPERATING_HOURS
TOTAL_SLOT_CAPACITY = SLOTS_PER_DAY * NUM_SERVERS * SIMULATION_DAYS
TOTAL_PATIENT_POOL = int(TOTAL_SLOT_CAPACITY * (1 + EXTRA_DEMAND_RATE))

print(f"Slots per day: {SLOTS_PER_DAY}")
print(f"Total slot capacity per day: {TOTAL_SLOT_CAPACITY}")
print(f"Total patient pool for simulation: {TOTAL_PATIENT_POOL}")


In [ ]:
# ============================================================
# #                  OVERBOOKING AND POLICY                  #
# ============================================================

RULE_NAME = "simple_pairing"
OVERBOOKING_LEVEL = 12

In [ ]:
# ============================================================
# #                  SIMULATION PARAMETERS                   #
# ============================================================

PROTECTED_GROUP_PROPORTION = 0.5
NUM_REPLICAS = 35
NUM_ITERATIONS = 30

THRESHOLDS = [
    (0.05,0.05),
    (0.07,0.07),
    (0.1, 0.1),
    (0.12, 0.12),
    (0.15, 0.15),
    (0.17, 0.17),
    (0.2, 0.2),
    (0.22, 0.22),
    (0.25, 0.25),
]

# Print thresholds
print("Thresholds for evaluation:")
for idx, (thr_1, thr_2) in enumerate(THRESHOLDS):
    print(f"Thresholds {idx + 1}: Protected = {thr_1}, Unprotected = {thr_2}")

### Initialize patients and load model probabilities

In [ ]:
X_file_path = project_path / "prediction_model/X.npy"
y_file_path = project_path / "prediction_model/y.npy"
column_file_path =  project_path / "prediction_model/column_names.pkl"

with open(column_file_path, 'rb') as file:
    column_names = pickle.load(file)

X = np.load(X_file_path, allow_pickle=True)
y = np.load(y_file_path, allow_pickle=True)

X = pd.DataFrame(X)
X.columns = column_names

patients_original = [Patient(i, **row) for i, row in enumerate(X.to_dict(orient='records'))]

In [ ]:
patient_proba_path = project_path / "patients_with_proba.pkl"

with open(patient_proba_path, 'rb') as f:
    patients_with_proba = pickle.load(f)

patients = patients_with_proba.copy()
probas = [patient.proba for patient in patients]

ppv_npv_results_file_path = "prediction_model/ppv_npv_results.xlsx"
ppv_npv_df = pd.read_excel(ppv_npv_results_file_path)

In [ ]:
# Splitting patients into cohorts
protected_group = [p for p in patients if p.protected]
general_group = [p for p in patients if not p.protected]

# Calculating requirements based on constants
required_protected = int(TOTAL_PATIENT_POOL * PROTECTED_GROUP_PROPORTION)
required_general = TOTAL_PATIENT_POOL - required_protected

# Validation Checks
is_protected_pool_sufficient = len(protected_group) >= required_protected

print(f"--- Population Summary ---")
print(f"Total Available Patients: {len(patients)}")
print(f"Protected Group Size: {len(protected_group)}")
print(f"General Group Size: {len(general_group)}")
print(f"--- Sampling Requirements ---")
print(f"Target Sample Size: {TOTAL_PATIENT_POOL}")
print(f"Protected Patients Needed: {required_protected}")
print(f"General Patients Needed: {required_general}")
print(f"Sufficient Protected Patients: {is_protected_pool_sufficient}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from scipy.stats import ks_2samp

prot_probas = [p.proba for p in patients if p.protected]
nonprot_probas = [p.proba for p in patients if not p.protected]

print(f"Protected proba: mean={np.mean(prot_probas):.4f}, std={np.std(prot_probas):.4f}")
print(f"NonProt proba:   mean={np.mean(nonprot_probas):.4f}, std={np.std(nonprot_probas):.4f}")

# Distribution comparison
stat, pval = ks_2samp(prot_probas, nonprot_probas)
print(f"KS test: statistic={stat:.4f}, p-value={pval:.4g}")

# Side-by-side plots
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)

# Protected
sns.histplot(
    prot_probas,
    color="blue",
    kde=True,
    stat="density",
    bins=30,
    alpha=0.6,
    ax=axes[0]
)
axes[0].set_title("Protected")
axes[0].set_xlabel("Probability")
axes[0].set_ylabel("Density")
axes[0].grid(True, alpha=0.3)

# Non-Protected
sns.histplot(
    nonprot_probas,
    color="red",
    kde=True,
    stat="density",
    bins=30,
    alpha=0.6,
    ax=axes[1]
)
axes[1].set_title("Non-Protected")
axes[1].set_xlabel("Probability")
axes[1].grid(True, alpha=0.3)

plt.suptitle("Probability Distributions Comparison", fontsize=14)
plt.tight_layout()
plt.show()

### Simulation

In [ ]:
files_prefix = f"{RULE_NAME}_overbooking{OVERBOOKING_LEVEL}_sample{TOTAL_PATIENT_POOL}_prot{PROTECTED_GROUP_PROPORTION:.2f}"

DEBUG_VERBOSE = False
model = None

results_summary_rows = []

metric_rows = {
    "PatientWT": [],
    "CRC_Rate": [],
    "CR_Rate": [],
    "IdleTime": [],
    "OverTime": [],
    "OverallWT": [],
    "TotalAttended": [],
}

for threshold_iter, thresholds in enumerate(THRESHOLDS):

    clear_output(wait=True)

    threshold_protected = thresholds[0]
    threshold_no_protected = thresholds[1]

    protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]["protected_ppv"].values[0]
    non_protected_ppv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]["unprotected_ppv"].values[0]
    protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_protected]["protected_npv"].values[0]
    non_protected_npv = ppv_npv_df[ppv_npv_df["threshold"] == threshold_no_protected]["unprotected_npv"].values[0]

    with open("patients_with_proba.pkl", "rb") as f:
        patients_og = pickle.load(f)

    all_measures = []
    convergence_values = []

    print(
        f"Simulating PT {threshold_protected} & NPT {threshold_no_protected} | "
        f"[P PPV {protected_ppv:.4f}] [NP PPV {non_protected_ppv:.4f}] "
        f"[P NPV {protected_npv:.4f}] [NP NPV {non_protected_npv:.4f}]"
    )

    # ============================================================
    # MONTE CARLO LOOP
    # ============================================================
    convergence = False
    iterations  = 0

    while not convergence and iterations < NUM_REPLICAS:

        start = time.time()
        iterations += 1

        patients = copy.deepcopy(patients_og)
        created_appointments = create_appointments(NUM_SERVERS, SIMULATION_DAYS, OPERATING_HOURS, SLOT_DURATION_MINS)

        sampled_random_patient_list = random_patient_sample(patients, TOTAL_PATIENT_POOL, PROTECTED_GROUP_PROPORTION, SIMULATION_DAYS)

        appointments_from_rule, num_refused = call_a_rule(
            sampled_random_patient_list, created_appointments,
            RULE_NAME, model,
            threshold_protected, threshold_no_protected,
            overbooking_level=OVERBOOKING_LEVEL,
        )

        for inner_iter in range(NUM_ITERATIONS):

            appointments = copy.deepcopy(appointments_from_rule)
            random_patient_list = copy.deepcopy(sampled_random_patient_list)

            attendance_random_patient_list = establish_attendance(
                random_patient_list,
                protected_ppv, non_protected_ppv, protected_npv, non_protected_npv,
                threshold_protected, threshold_no_protected,
            )

            clinic = Clinic(attendance_random_patient_list, appointments, SLOT_DURATION_MINS)
            clinic.simulation()

            measures = clinic.get_measures()

            # Flatten per-server lists into single scalars for easier aggregation later.
            measures["idle_time_total"] = float(sum(measures["idle_time_server"]))
            measures["over_time_total"] = float(sum(measures["over_time"]))

            all_measures.append(measures)

            del appointments, random_patient_list, attendance_random_patient_list, clinic

        if iterations >= 20:

            tmp_df = pd.DataFrame(all_measures).copy()
            tmp_df["replica_id"] = np.repeat(np.arange(iterations), NUM_ITERATIONS)[:len(tmp_df)]

            replica_records = (
                tmp_df.drop(columns=["idle_time_server", "over_time"], errors="ignore")
                .groupby("replica_id")
                .mean(numeric_only=True)
                .to_dict(orient="records")
            )

            convergence, diff = check_convergence_mean(replica_records, converge_mean=0.02, verbose=DEBUG_VERBOSE)
            convergence_values.append(diff)

        end = time.time()
        print(f"Ran Successfully Replica {iterations}/{NUM_REPLICAS} in {end-start:.2f}s")

        del created_appointments, patients

    print("* * * Ran Successfully All Replicas!!!")

    measures_df = pd.DataFrame(all_measures)
    measures_df["replica_id"] = np.repeat(np.arange(iterations), NUM_ITERATIONS)[:len(measures_df)]

    prot_anchor = sum(1 for p in sampled_random_patient_list if p.protected and not p.overbooked_target and p.assigned)
    nonprot_anchor = sum(1 for p in sampled_random_patient_list if not p.protected and not p.overbooked_target and p.assigned)
    prot_flagged = sum(1 for p in sampled_random_patient_list if p.protected and p.overbooked_target and p.assigned)
    nonprot_flagged = sum(1 for p in sampled_random_patient_list if not p.protected and p.overbooked_target and p.assigned)

    prot_total = prot_anchor + prot_flagged
    nonprot_total = nonprot_anchor + nonprot_flagged
    prot_flagged_pct = (prot_flagged / prot_total) if prot_total > 0 else 0
    nonprot_flagged_pct = (nonprot_flagged / nonprot_total) if nonprot_total > 0 else 0

    replica_grouped = (
        measures_df
        .groupby("replica_id")
        .agg(
            prot_total_wt = ("clients_total_waiting_time protected class", "sum"),
            nonprot_total_wt = ("clients_total_waiting_time non protected class", "sum"),
            prot_attend = ("protected_assistance", "sum"),
            nonprot_attend = ("non_protected_assistance", "sum"),
            crc_prot    = ("crc_protected",     "sum"),
            crc_nonprot = ("crc_non_protected", "sum"),
            cr_prot     = ("cr_protected",      "sum"),
            cr_nonprot  = ("cr_non_protected",  "sum"),
            total_wt = ("total_waiting_time", "sum"),
            total_attended = ("total_attended_patients", "sum"),
            idle_time = ("idle_time_total", "sum"),
            over_time = ("over_time_total", "sum"),
        )
        .reset_index()
    )

    # Operational metrics — daily averages per realization
    idle_time_replica = replica_grouped["idle_time"] / (NUM_ITERATIONS * SIMULATION_DAYS)
    over_time_replica = replica_grouped["over_time"] / (NUM_ITERATIONS * SIMULATION_DAYS)
    total_attended_replica = replica_grouped["total_attended"] / (NUM_ITERATIONS * SIMULATION_DAYS)

    # Overall patient WT — ratio of sums (correct aggregation)
    overall_patient_wt_replica = safe_divide(replica_grouped["total_wt"], replica_grouped["total_attended"]).dropna()

    # Per-group mean WT — headline fairness metric
    prot_mean_wt_replica = safe_divide(replica_grouped["prot_total_wt"], replica_grouped["prot_attend"]).dropna()
    nonprot_mean_wt_replica = safe_divide(replica_grouped["nonprot_total_wt"], replica_grouped["nonprot_attend"]).dropna()

    crc_rate_prot_replica = safe_divide(replica_grouped["crc_prot"], replica_grouped["prot_attend"]).dropna()
    crc_rate_nonprot_replica = safe_divide(replica_grouped["crc_nonprot"], replica_grouped["nonprot_attend"]).dropna()

    cr_rate_prot_replica = safe_divide(replica_grouped["cr_prot"], replica_grouped["prot_attend"]).dropna()
    cr_rate_nonprot_replica = safe_divide(replica_grouped["cr_nonprot"], replica_grouped["nonprot_attend"]).dropna()

    # Diagnostic: track effective sample size after dropna
    if DEBUG_VERBOSE:
        print(f"  [diagnostic] effective n_replicas: "
            f"prot_wt={len(prot_mean_wt_replica)}, nonprot_wt={len(nonprot_mean_wt_replica)}, "
            f"crc_prot={len(crc_rate_prot_replica)}, crc_nonprot={len(crc_rate_nonprot_replica)}, "
            f"cr_prot={len(cr_rate_prot_replica)}, cr_nonprot={len(cr_rate_nonprot_replica)}")

    idle_mean, idle_margin = confidence_interval(idle_time_replica)
    over_mean, over_margin = confidence_interval(over_time_replica)
    overall_wt_mean, overall_wt_margin = confidence_interval(overall_patient_wt_replica)
    total_att_mean, total_att_margin = confidence_interval(total_attended_replica)

    prot_wt_mean, prot_wt_margin = confidence_interval(prot_mean_wt_replica)
    nonprot_wt_mean, nonprot_wt_margin = confidence_interval(nonprot_mean_wt_replica)

    crc_rate_prot_mean, crc_rate_prot_margin = confidence_interval(crc_rate_prot_replica)
    crc_rate_nonprot_mean, crc_rate_nonprot_margin = confidence_interval(crc_rate_nonprot_replica)

    cr_rate_prot_mean, cr_rate_prot_margin = confidence_interval(cr_rate_prot_replica)
    cr_rate_nonprot_mean, cr_rate_nonprot_margin = confidence_interval(cr_rate_nonprot_replica)

    tests_patient_wt = run_metric_tests(prot_mean_wt_replica, nonprot_mean_wt_replica, "PatientWT")
    tests_crc = run_metric_tests(crc_rate_prot_replica, crc_rate_nonprot_replica, "CRC_Rate")
    tests_cr  = run_metric_tests(cr_rate_prot_replica,  cr_rate_nonprot_replica,  "CR_Rate")

    def _paired_row(prot_mean, prot_margin, nonprot_mean, nonprot_margin, tests_df):
        """Returns a flat dict summarising one threshold pair for a paired metric."""
        row = {
            "protected_threshold":     threshold_protected,
            "non_protected_threshold": threshold_no_protected,
            "replicas":                iterations,
            "inner_iterations":        NUM_ITERATIONS,
            "protected_ppv":           protected_ppv,
            "non_protected_ppv":       non_protected_ppv,
            "protected_npv":           protected_npv,
            "non_protected_npv":       non_protected_npv,
            "prot_mean":               prot_mean,
            "prot_ci_lower":           prot_mean - prot_margin,
            "prot_ci_upper":           prot_mean + prot_margin,
            "nonprot_mean":            nonprot_mean,
            "nonprot_ci_lower":        nonprot_mean - nonprot_margin,
            "nonprot_ci_upper":        nonprot_mean + nonprot_margin,
        }
        # Flatten hypothesis-test columns (one column per test row x field)
        for _, tr in tests_df.iterrows():
            prefix = f"{tr['metric']}_{tr['alternative']}"
            row[f"{prefix}_t_stat"]  = tr["t_stat"]
            row[f"{prefix}_p_value"] = tr["p_value"]
            row[f"{prefix}_welch"]   = tr["welch_correction"]
        return row

    def _single_row(mean_val, margin, series):
        """Returns a flat dict summarising one threshold pair for a single-group metric."""
        return {
            "protected_threshold":     threshold_protected,
            "non_protected_threshold": threshold_no_protected,
            "replicas":                iterations,
            "inner_iterations":        NUM_ITERATIONS,
            "protected_ppv":           protected_ppv,
            "non_protected_ppv":       non_protected_ppv,
            "protected_npv":           protected_npv,
            "non_protected_npv":       non_protected_npv,
            "mean":                    mean_val,
            "ci_lower":                mean_val - margin,
            "ci_upper":                mean_val + margin,
            "n_replicas":              len(series),
        }

    metric_rows["PatientWT"].append(
        _paired_row(prot_wt_mean, prot_wt_margin,
                    nonprot_wt_mean, nonprot_wt_margin,
                    tests_patient_wt))

    metric_rows["CRC_Rate"].append(
        _paired_row(crc_rate_prot_mean, crc_rate_prot_margin,
                    crc_rate_nonprot_mean, crc_rate_nonprot_margin,
                    tests_crc))

    metric_rows["CR_Rate"].append(
        _paired_row(cr_rate_prot_mean, cr_rate_prot_margin,
                    cr_rate_nonprot_mean, cr_rate_nonprot_margin,
                    tests_cr))

    metric_rows["IdleTime"].append(
        _single_row(idle_mean, idle_margin, idle_time_replica))

    metric_rows["OverTime"].append(
        _single_row(over_mean, over_margin, over_time_replica))

    metric_rows["OverallWT"].append(
        _single_row(overall_wt_mean, overall_wt_margin, overall_patient_wt_replica))

    metric_rows["TotalAttended"].append(
        _single_row(total_att_mean, total_att_margin, total_attended_replica))

    summary_row = {
        "rule": RULE_NAME,
        "protected_threshold": threshold_protected,
        "non_protected_threshold": threshold_no_protected,
        "replicas": iterations,
        "inner_iterations": NUM_ITERATIONS,
        "protected_ppv": protected_ppv,
        "non_protected_ppv": non_protected_ppv,
        "protected_npv": protected_npv,
        "non_protected_npv": non_protected_npv,
        "idle_time": idle_mean,
        "over_time": over_mean,
        "overall_patient_wt": overall_wt_mean,
        "total_attended": total_att_mean,
        "protected_patient_wt": prot_wt_mean,
        "non_protected_patient_wt": nonprot_wt_mean,

        "protected_crc_rate":          crc_rate_prot_mean,
        "non_protected_crc_rate":      crc_rate_nonprot_mean,
        "protected_cr_rate":           cr_rate_prot_mean,
        "non_protected_cr_rate":       cr_rate_nonprot_mean,


        "protected_anchors":          prot_anchor,
        "protected_flagged":          prot_flagged,
        "non_protected_anchors":      nonprot_anchor,
        "non_protected_flagged":      nonprot_flagged,
        "protected_flagged_pct":      prot_flagged_pct,
        "non_protected_flagged_pct":  nonprot_flagged_pct,
        "flagged_pct_asymmetry":      prot_flagged_pct - nonprot_flagged_pct,

    }

    results_summary_rows.append(summary_row)


# ============================================================
# #                       WRITE EXCEL                        #
# ============================================================

results_summary_df = pd.DataFrame(results_summary_rows)

base_dir = Path("simulation_outputs/excel_files/post_thesis_iterations")
base_dir.mkdir(parents=True, exist_ok=True)

summary_path = base_dir / f"{files_prefix}_{NUM_REPLICAS}_replicas_{NUM_ITERATIONS}_iter_summary.xlsx"

summary_path.unlink(missing_ok=True)

with pd.ExcelWriter(summary_path, engine="openpyxl") as writer:

    # Summary sheet — one row per threshold pair (unchanged)
    results_summary_df.to_excel(writer, sheet_name="Summary", index=False)
    autosize_excel_columns(writer, "Summary")

    # One sheet per metric — one row per threshold pair
    for metric_name, rows in metric_rows.items():
        df_metric = pd.DataFrame(rows)
        sheet_name = metric_name[:31]
        df_metric.to_excel(writer, sheet_name=sheet_name, index=False)
        autosize_excel_columns(writer, sheet_name)

print(f"[summary] Saved {len(results_summary_rows)} rows -> {summary_path}")